# Run the HydroCNHS FE Walter DKP example

## Goal
This repository runs hydrocnhs.Model.run, the Dam API, shared physics and downstream routing using frozen inflow. The DKP decision adapter adds retrieved knowledge to FE Walter public facts and the current daily state. The preserved builder retrieves relevant document sections; it does not paste the whole DKP or fine-tune a model. The actual system prompt and DKP remain separate files.

## Setup
Python 3.10+ with repository dependencies. All cells use local files; no provider calls.

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: clean-upgrade-99ea43e42a


## Steps

### Inspect the prompt and knowledge entry

In [2]:
print((ROOT/'examples/fe_walter/dkp/experiment.tutorial.yaml').read_text())
print((ROOT/'prompts/fe_walter_reference/dkp_system.txt').read_text())
dkp=(ROOT/'data/anik_v3_7/DKP_FEWalter_Dam_v1.2.md').read_text()
print('DKP headings:', [s for s in dkp.splitlines() if s.startswith('#')][:12])

# FE Walter V3.7 dkp: original prompt builders, synthetic replies, zero model calls.
schema_version: llm-reservoir-experiment.v1
case: ../../../cases/fe_walter_reference/case.json # Frozen hydrology sample; V3.7 physics.
run_id: fe-walter-tutorial-dkp        # Change with output before a new run.
output: ../../../runs/fe-walter-tutorial-dkp
max_calls: 0                             # This example cannot call a provider.
auditors:
  release_limits:
    type: release_bounds
    version: "1"
agents:
  fe_operator:
    reservoir: FEWalter                  # Reservoir ID in case.json.
    mode: offline                        # Native prompt/audit run on fixture replies.
    strategy: reference_v3_7             # Generic strategy selector; optional vendor plugin.
    reference:
      profile: fe_walter                 # This reference profile is FE-specific.
      method: dkp                   # Original zero-shot or DKP policy.
      system_prompt: ../../../prompts/fe_walter_reference/dkp_sy

### Run and inspect retrieval evidence

In [3]:
settings, verified=ensure_offline('examples/fe_walter/dkp/experiment.tutorial.yaml')
print(json.dumps(verified,indent=2))
raw=[json.loads(p.read_text()) for p in (Path(settings.report['output'])/'raw').glob('policy_*.json')]
first=min(raw,key=lambda item:item['backend_trace']['input_variables']['timestep'])
prompt=first['backend_trace']['prompt']
assert 'Retrieved DKP sections' in prompt
assert dkp not in prompt
print('Retrieved excerpt, first 1200 characters:',prompt[prompt.index('Retrieved DKP sections'):][:1200])
print('Native audit retained:', 'backend_audit' in first)


{
  "status": "PASS",
  "days": 3,
  "reservoir_count": 1,
  "reservoir_days": 3,
  "date_start": "2016-01-01",
  "date_end": "2016-01-03",
  "audit_counts": {
    "PASS": 3
  },
  "max_reservoir_balance_residual_m3": 0.0,
  "max_system_balance_residual_m3": 0.0,
  "max_routing_difference_cms": 0.0,
  "final_channel_storage_m3": 0.0,
  "observed_network_calls": 0,
  "operating_strategy_accepted": null,
  "audit_binding_counts": {
    "versioned_suite": 3,
    "legacy_unbound": 0
  },
  "external_audit_pins_checked": []
}
Retrieved excerpt, first 1200 characters: Retrieved DKP sections relevant to current conditions:

--- DKP excerpt 1: R-RELEASE (composite per-timestep decision; WCM Table 7-1 Parts A and B; §7-01, §7-03, §7-05) ---
```pseudo
plan = R-PLAN(pool_elevation, stages, season)

active_min = R-MINFC       if plan in {C, D} or impounding_for_flood_emergency   # 100 cfs
           | R-MINDROUGHT  if drbc_drought_declared                              #  43 cfs
           | R-MIN

## Checks
Review the printed inputs, counts, source labels and residuals. Synthetic replies validate interfaces only.

## Next steps
The native V3.7 DKP builder/auditor is FE-specific. Other reservoirs use the configured prompt/knowledge/auditor path with their own profile and rules; merely renaming FE Walter is rejected.